# Environment configuration — bronze/vlp
Credentials are external. Set PIPELINE_ENV and job variables generated from config/environments and config/secrets.
This repository revision changes configuration handling, not the source ingestion/write algorithms.


# NB_bronze.vlp.order_detail
Landing JSON → Bronze Iceberg, giữ cấu trúc cell và cấu hình kết nối theo notebook mẫu HRM.
Landing: `s3://landing-bucket/vlp/order_detail/YYYYMMDD/`. Bảng: `bronze.vlp.order_detail`.


## Import, cấu hình và chạy
1. CP4D → Projects → DataLakehouse → Assets → New asset → Notebook → Local file.
2. Landing dùng Python với requests/boto3 (không cần Spark). Bronze dùng Python 3.11 with Spark và connector watsonx.data như notebook HRM mẫu. Restart kernel và Run All sau khi import.
3. Cấu hình S3/metastore/Spark giữ từ mẫu. API mặc định UAT và test secret trong spec. Không dùng sign cố định của curl: chữ ký được tính lại theo toàn bộ tham số bên trong, kể cả số trang.
4. Daily: bỏ các biến VLP_DATE_FROM/VLP_DATE_TO, ETL_Date mặc định T−1 Việt Nam; request theo ngày tạo từ 00:00 đến trước 00:00 hôm sau. Spec mô tả createTimeEnd inclusive nên gửi cuối ngày = cận trên trừ 1 đơn vị.
5. Initial: truyền ETL_Date=ngày lưu tập khởi tạo, VLP_DATE_FROM=2020-01-01, VLP_DATE_TO=ngày kết thúc exclusive. Ví dụ ETL_Date=2026-10-21, VLP_DATE_FROM=2020-01-01, VLP_DATE_TO=2026-10-22. Một tập init được lưu vào partition batch, không tái dựng partition lịch sử từng ngày.
6. Landing lấy và kiểm tra đủ mọi trang trong thư mục tạm local; xóa file cũ đúng vlp/order_detail/YYYYMMDD/, upload trang mới, chốt manifest cuối cùng. Lỗi API trước bước xóa giữ tập cũ; lỗi ghi file không có manifest hoàn tất. Batch rỗng hợp lệ vẫn thay thế ngày.
7. Bronze dùng cùng ETL_Date, tự đọc manifest; MANIFEST_PATH và LANDING_RUN_ID là tùy chọn để khóa đúng tập đầu vào. Giữ nguyên nested data dưới dạng JSON string trong cột tương ứng và raw_record; không explode parcel/track làm nhân bản order. Ghi atomic overwrite partition ngày; log tại bronze.pipeline.log_pipeline.
8. Chỉ chạy Bronze sau khi Landing SUCCESS. Không chạy lại Landing cùng ngày trong khi Bronze đang đọc. Chỉ một job xử lý cùng nguồn/object/ngày; tổng tốc độ các job cùng gateway phải dưới giới hạn 10 request/giây. Mặc định một job gọi 5 request/giây.

## Phạm vi và điểm cần kiểm tra trên UAT
- Spec chỉ có createTimeStart/createTimeEnd, chưa có bộ lọc cập nhật gmtModified. Daily hiện lấy đơn được tạo trong ngày; không bảo đảm bắt thay đổi/tracking mới của các đơn cũ. Cần API lọc cập nhật hoặc cơ chế quét lại/đối soát khi bổ sung yêu cầu đó.
- createTimeStart/createTimeEnd có kiểu Long nhưng spec chưa ghi đơn vị. Notebook mặc định epoch milliseconds, cho phép VLP_TIME_UNIT=seconds. Kiểm tra với một đơn biết gmtCreate và thời gian đúng trước khi lập lịch; notebook kiểm tra mọi gmtCreate trong response thuộc cửa sổ request.
- Phân trang dùng currentPage/pageSize, response data.paging.totalCount. Tổng thay đổi, lặp trang, trùng orderCode hoặc thiếu trang đều dừng trước khi thay thế dữ liệu ngày. Pagination khớp không chứng minh nguồn có snapshot nhất quán giữa các trang.
- Mỗi trang giữ trong RAM; toàn bộ dữ liệu tạm/khóa chống trùng lưu trên disk driver. Bronze stage NDJSON lên S3 để Spark đọc phân tán; xóa stage sau khi hoàn tất/lỗi. Cần đủ disk tạm theo volume ngày/init.
- Chưa gọi API hoặc chạy Spark/metastore thực tế; việc giữ cấu hình mẫu không tự sửa lỗi metastore đã gặp ở HRM.


In [ ]:
# Chạy một lần khi bắt đầu, theo mẫu.
if "spark" in globals() and spark is not None: spark.stop()


In [ ]:
import os, sys, json, time, logging, re, uuid, hashlib, tempfile, sqlite3
from datetime import datetime, date, timedelta, timezone
from zoneinfo import ZoneInfo
from urllib.parse import urlparse
import boto3
from botocore.config import Config
from pyspark.sql import SparkSession, functions as F, types as T


In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", stream=sys.stdout, force=True)
logger = logging.getLogger("vlp_ingest")

In [ ]:
start_date = datetime.now(timezone.utc)
status, log_message, record_count, df = "Fail", None, 0, None

In [ ]:
# Environment configuration is injected by CP4D/job environment variables.
# Generate a job env file with scripts/build_job_env.py. Never embed credentials here.
def _need_env(name):
    value = os.getenv(name, "").strip()
    if not value:
        raise ValueError("Thiếu biến môi trường: " + name)
    return value

PIPELINE_ENV = _need_env("PIPELINE_ENV")
if PIPELINE_ENV not in ("dev", "uat", "prod"):
    raise ValueError("PIPELINE_ENV phải là dev, uat hoặc prod")

# 1. Lấy các biến môi trường
now = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh"))
yesterday = now - timedelta(days=1)

# 3. Định dạng lại thành chuỗi YYYYMMDD
yesterday = yesterday.strftime("%Y-%m-%d")

#S3
S3_ENDPOINT = _need_env("S3_ENDPOINT")
ACCESS_KEY = _need_env("ACCESS_KEY")
SECRET_KEY = _need_env("SECRET_KEY")
BUCKET_NAME = _need_env("BUCKET_NAME")

#Metastore
WXD_HMS_USERNAME = _need_env("WXD_HMS_USERNAME")
WXD_HMS_PASSWORD = _need_env("WXD_HMS_PASSWORD")
ICEBERG_CATALOG1 = os.getenv("ICEBERG_CATALOG1", "bronze")
BUCKET_NAME1 = _need_env("BUCKET_NAME1")
ACCESS_KEY1 = os.getenv("ACCESS_KEY1") or ACCESS_KEY
SECRET_KEY1 = os.getenv("SECRET_KEY1") or SECRET_KEY

HIVEMETASTORE_HOST = _need_env("HIVEMETASTORE_HOST")

HIVE_TRUSTSTORE_PATH = _need_env("HIVE_TRUSTSTORE_PATH")
HIVE_TRUSTSTORE_PASSWORD = _need_env("HIVE_TRUSTSTORE_PASSWORD")

WXD_INSTANCE_ID = _need_env("WXD_INSTANCE_ID")
HADOOP_AMS_URL = _need_env("HADOOP_AMS_URL")
HADOOP_CAS_ENDPOINT = _need_env("HADOOP_CAS_ENDPOINT")
HADOOP_CPG_ENDPOINT = _need_env("HADOOP_CPG_ENDPOINT")


TABLE_NAME=os.getenv("TABLE_NAME","bronze.vlp.order_detail")
SOURCE_NAME=os.getenv("SOURCE_NAME","vlp.api.order_detail")
STEP='landing_2_bronze'


ETL_Date = os.getenv("ETL_Date",yesterday)
TZ = ZoneInfo("Asia/Ho_Chi_Minh")
BATCH_DATE = date.fromisoformat(ETL_Date)
if ETL_Date != BATCH_DATE.isoformat(): raise ValueError("ETL_Date phải đúng YYYY-MM-DD")
DAY_PREFIX = f"vlp/order_detail/{BATCH_DATE.strftime('%Y%m%d')}"
LOG_PREFIX = f"vlp/order_detail/_logs/{BATCH_DATE.strftime('%Y%m%d')}"
LOG_TABLE = os.getenv("LOG_TABLE", "bronze.pipeline.log_pipeline")
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", ICEBERG_CATALOG1): raise ValueError("Catalog không hợp lệ")
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", TABLE_NAME):
    raise ValueError("TABLE_NAME phải là catalog.schema.table")
RUN_ID = uuid.uuid4().hex

SOURCE_NAME = os.getenv("SOURCE_NAME", f"s3://{BUCKET_NAME}/{DAY_PREFIX}")


In [ ]:
def init_spark():
    spark = SparkSession.builder \
        .appName(f"import_table") \
        .config("spark.sql.extensions","org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG1}","org.apache.iceberg.spark.SparkCatalog") \
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG1}.type", "hive") \
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG1}.uri",
                f"{HIVEMETASTORE_HOST}?catalog={ICEBERG_CATALOG1}") \
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG1}.warehouse",
                f"s3a://{BUCKET_NAME1}") \
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG1}.hive.metastore.client.plain.username",
                WXD_HMS_USERNAME) \
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG1}.hive.metastore.client.plain.password",
                WXD_HMS_PASSWORD) \
        .config("spark.hive.metastore.uris",HIVEMETASTORE_HOST) \
        .config("spark.hive.metastore.use.SSL", "true") \
        .config("spark.hive.metastore.truststore.type", "JKS") \
        .config("spark.hive.metastore.truststore.path", HIVE_TRUSTSTORE_PATH) \
        .config("spark.hive.metastore.truststore.password", HIVE_TRUSTSTORE_PASSWORD) \
        .config("spark.hive.metastore.client.auth.mode", "PLAIN") \
        .config("spark.hive.metastore.client.plain.username", WXD_HMS_USERNAME) \
        .config("spark.hive.metastore.client.plain.password", WXD_HMS_PASSWORD) \
        .config("spark.hive.metastore.client.thrift.transport.mode", "http") \
        .config("spark.hadoop.wxd.ams.url", HADOOP_AMS_URL) \
        .config("spark.hadoop.wxd.cas.endpoint", HADOOP_CAS_ENDPOINT) \
        .config("spark.hadoop.wxd.cpg.endpoint", HADOOP_CPG_ENDPOINT) \
        .config("spark.hadoop.wxd.instanceId", WXD_INSTANCE_ID) \
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
        .config("spark.hadoop.fs.s3a.path.style.access", "true") \
        \
        .config("spark.hadoop.fs.s3a.endpoint", S3_ENDPOINT) \
        .config("spark.hadoop.fs.s3a.path.style.access", "true") \
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "true") \
        .config("spark.hadoop.fs.s3a.access.key", ACCESS_KEY) \
        .config("spark.hadoop.fs.s3a.secret.key", SECRET_KEY) \
        .config("spark.hadoop.fs.s3a.aws.credentials.provider","org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
        \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.endpoint", S3_ENDPOINT) \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.access.key", ACCESS_KEY) \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.secret.key", SECRET_KEY) \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.aws.credentials.provider","org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.path.style.access", "true") \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.connection.ssl.enabled", "true") \
        \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.endpoint", S3_ENDPOINT) \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.access.key", ACCESS_KEY1) \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.secret.key", SECRET_KEY1) \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.aws.credentials.provider","org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.path.style.access", "true") \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.connection.ssl.enabled", "true") \
        \
        .config("spark.hadoop.iceberg.engine.hive.lock-enabled", "false") \
        .config("spark.sql.catalogImplementation", "hive") \
        .getOrCreate()


    spark.sparkContext.setLogLevel("WARN")
    return spark

In [ ]:
def required(name):
    value = os.getenv(name, "").strip()
    if not value:
        raise ValueError("Thiếu cấu hình: " + name)
    return value

def pack(obj):
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False).encode("utf-8")

def init_s3():
    return boto3.client("s3", endpoint_url=S3_ENDPOINT,
        aws_access_key_id=ACCESS_KEY, aws_secret_access_key=SECRET_KEY,
        config=Config(signature_version="s3v4", connect_timeout=15, read_timeout=120,
                      retries={"max_attempts": 3, "mode": "standard"},
                      s3={"addressing_style": "path"}))

def put_json(key, obj):
    blob = pack(obj)
    s3.put_object(Bucket=BUCKET_NAME, Key=key, Body=blob, ContentType="application/json")
    return hashlib.sha256(blob).hexdigest()

def get_json(key):
    return json.loads(s3.get_object(Bucket=BUCKET_NAME, Key=key)["Body"].read())

def delete_prefix(prefix):
    # Chỉ cho phép đúng partition ngày hoặc thư mục stage của run hiện tại.
    allowed = {DAY_PREFIX + "/", f"vlp/order_detail/_bronze_stage/{RUN_ID}/"}
    if prefix not in allowed:
        raise ValueError("Từ chối xóa ngoài phạm vi job")
    keys = []
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET_NAME, Prefix=prefix):
        keys.extend(x["Key"] for x in page.get("Contents", []))
    # Xóa manifest trước để không chốt tập file đang thay thế.
    manifest_key = prefix + "manifest.json"
    if manifest_key in keys:
        keys.remove(manifest_key)
        keys.insert(0, manifest_key)
        result = s3.delete_objects(Bucket=BUCKET_NAME, Delete={"Objects": [{"Key": manifest_key}]})
        if result.get("Errors"):
            raise RuntimeError("Xóa manifest không thành công")
        keys.remove(manifest_key)
    for i in range(0, len(keys), 1000):
        result = s3.delete_objects(Bucket=BUCKET_NAME, Delete={"Objects": [{"Key": k} for k in keys[i:i+1000]]})
        if result.get("Errors"):
            raise RuntimeError("Xóa file S3 không thành công")

def error_summary(exc):
    if isinstance(exc, ValueError) and not isinstance(exc, json.JSONDecodeError):
        return str(exc)
    if isinstance(exc, VlpApiError):
        return str(exc)
    response = getattr(exc, "response", None)
    if isinstance(response, dict):
        return "S3 error: " + str(response.get("Error", {}).get("Code", "unknown"))
    return type(exc).__name__ + ": kiểm tra kết nối/runtime; không in credentials hoặc dữ liệu đơn hàng."

class VlpApiError(RuntimeError):
    pass

def decode_page(payload, page, requested_size):
    if not isinstance(payload, dict):
        raise ValueError("Response VLP không phải object")
    if payload.get("success") is not True:
        # Không log response/errorMessage vì có thể chứa request hoặc dữ liệu cá nhân.
        code = re.sub(r"[^A-Za-z0-9_.-]", "", str(payload.get("errorCode", "unknown")))[:100]
        trace = re.sub(r"[^A-Za-z0-9_.-]", "", str(payload.get("traceId", "unknown")))[:100]
        raise VlpApiError(f"VLP success != true; errorCode={code}; traceId={trace}")
    data = payload.get("data")
    if not isinstance(data, dict) or not isinstance(data.get("paging"), dict):
        raise ValueError("Thiếu data.paging")
    rows, paging = data.get("tableData"), data["paging"]
    if not isinstance(rows, list):
        raise ValueError("data.tableData không phải danh sách")
    for k in ("currentPage", "pageSize", "totalCount"):
        if type(paging.get(k)) is not int:
            raise ValueError("Metadata phân trang không phải integer: " + k)
    if paging["currentPage"] != page or paging["pageSize"] <= 0 or paging["totalCount"] < 0:
        raise ValueError("Metadata phân trang không hợp lệ")
    if paging["pageSize"] != requested_size or len(rows) > paging["pageSize"]:
        raise ValueError("pageSize trả về khác request hoặc số dòng vượt pageSize; điều chỉnh VLP_PAGE_SIZE theo server")
    for row in rows:
        if not isinstance(row, dict) or not isinstance(row.get("orderCode"), str) or not row["orderCode"]:
            raise ValueError("Bản ghi thiếu orderCode")
    return rows, paging

def validate_manifest(manifest):
    if (manifest.get("manifest_version") != 1 or manifest.get("source") != "vlp"
        or manifest.get("object") != "order_detail" or manifest.get("data_date") != ETL_Date
        or manifest.get("complete") is not True):
        raise ValueError("Manifest sai nguồn/object/ngày hoặc chưa chốt")
    if not re.fullmatch(r"[0-9a-f]{32}", manifest.get("run_id", "")):
        raise ValueError("Manifest run_id không hợp lệ")
    if type(manifest.get("count")) is not int or manifest["count"] < 0:
        raise ValueError("Manifest count không hợp lệ")
    files = manifest.get("files")
    if not isinstance(files, list) or not files:
        raise ValueError("Manifest thiếu danh sách file")
    for i, item in enumerate(files, 1):
        if item.get("key") != f"{DAY_PREFIX}/page-{i:06d}.json":
            raise ValueError("Manifest sai đường dẫn/thứ tự trang")
        if not re.fullmatch(r"[0-9a-f]{64}", item.get("sha256", "")):
            raise ValueError("Manifest thiếu checksum")
        if type(item.get("count")) is not int or item["count"] < 0:
            raise ValueError("Manifest page count không hợp lệ")
    if sum(x["count"] for x in files) != manifest["count"]:
        raise ValueError("Tổng số dòng manifest không khớp")

def seen_order(connection, key):
    try:
        connection.execute("INSERT INTO seen VALUES (?)", (key,))
    except sqlite3.IntegrityError:
        raise ValueError("orderCode trùng giữa/trong các trang; không chốt dữ liệu") from None


In [ ]:
BUSINESS_FIELDS = ['orderCode', 'orderStatus', 'orderStatusName', 'payStatus', 'payStatusName', 'interceptStatus', 'interceptStatusName', 'interceptReason', 'interceptReasonName', 'sourceOrderCode', 'orderChannel', 'orderChannelName', 'bizType', 'bizTypeName', 'relationOrderCode', 'orderType', 'orderDirection', 'orderDirectionName', 'payMethod', 'payMethodName', 'currency', 'customerType', 'customerTypeName', 'customerCode', 'customerName', 'contractCode', 'payer', 'payerName', 'couponCode', 'isCod', 'expressProductCode', 'expressProductName', 'valueAddedServiceList', 'collectionMethod', 'collectionMethodName', 'deliveryMethod', 'deliveryMethodName', 'expectCollectionTime', 'expectDeliveryTime', 'perfectCollectionPeriodName', 'perfectDeliveryPeriodName', 'volumeWeightRatio', 'remark', 'sender', 'recipient', 'senderPostman', 'recipientPostman', 'parcelCount', 'parcelList', 'waybillCodeList', 'externalWaybillCodeList', 'totalWeight', 'totalFee', 'totalNoTaxFee', 'totalTaxFee', 'taxRate', 'totalCodFee', 'orderFeeList', 'undeliverableAction', 'undeliverableActionName', 'undeliverableRetAddressType', 'undeliverableRetAddressTypeName', 'undeliverableRetAddress', 'govOfficePointCode', 'govOfficePointName', 'govProcedureTypeCode', 'govProcedureTypeName', 'govReceiptCode', 'govProcedureGroupCode', 'govProcedureGroupName', 'govAppFullName', 'govAppPhoneNumber', 'govAppIdType', 'govAppIdNumber', 'internationalTax', 'internationalDeliveryTerm', 'gmtCreate', 'gmtModified', 'creatorAccount', 'creatorName', 'modifierAccount', 'modifierName', 'actualCollectedTime', 'actualSignedTime', 'signFailedReason', 'signFailedReasonName', 'senderImgUrls', 'recipientImgUrls', 'secondLegFeeInfo']
def resolve_manifest():
    key = DAY_PREFIX + "/manifest.json"
    supplied = os.getenv("MANIFEST_PATH", "").strip()
    if supplied:
        parsed = urlparse(supplied)
        if parsed.scheme != "s3" or parsed.netloc != BUCKET_NAME or parsed.path.lstrip("/") != key or parsed.query or parsed.fragment:
            raise ValueError("MANIFEST_PATH phải trỏ đúng VLP object/ngày của job")
    manifest = get_json(key)
    validate_manifest(manifest)
    expected_run = os.getenv("LANDING_RUN_ID", "").strip()
    if expected_run and expected_run != manifest["run_id"]:
        raise ValueError("LANDING_RUN_ID không khớp tập Landing hiện tại")
    return manifest

def as_string(value):
    if value is None or isinstance(value, str): return value
    if isinstance(value, (list, dict, bool)): return pack(value).decode("utf-8")
    return str(value)

def normalize_order(row, manifest):
    record = {name: as_string(row.get(name)) for name in BUSINESS_FIELDS}
    record.update(raw_record=pack(row).decode("utf-8"), data_date=ETL_Date,
                  ingest_time=manifest["ingest_time"], run_id=RUN_ID, landing_run_id=manifest["run_id"])
    return record

def stage_landing(manifest):
    # Xử lý từng trang, dùng SQLite để bắt trùng khóa mà không giữ toàn bộ orders trong RAM.
    paths, total = [], 0
    stage_prefix = f"vlp/order_detail/_bronze_stage/{RUN_ID}/"
    with tempfile.TemporaryDirectory(prefix="vlp_bronze_") as tmp:
        with sqlite3.connect(os.path.join(tmp, "seen.sqlite")) as seen:
            seen.execute("CREATE TABLE seen (order_code TEXT PRIMARY KEY)")
            for page, item in enumerate(manifest["files"], 1):
                blob = s3.get_object(Bucket=BUCKET_NAME, Key=item["key"])["Body"].read()
                if hashlib.sha256(blob).hexdigest() != item["sha256"]:
                    raise ValueError("Checksum Landing khác manifest; tập ngày có thể đã bị thay thế")
                payload = json.loads(blob)
                rows, paging = decode_page(payload, page, manifest["page_size"])
                if len(rows) != item["count"] or paging["totalCount"] != manifest["count"]:
                    raise ValueError("Số dòng/trang trong Landing không khớp manifest")
                filename = f"part-{page:06d}.jsonl"
                local = os.path.join(tmp, filename)
                with open(local, "wb") as out:
                    for row in rows:
                        seen_order(seen, row["orderCode"])
                        out.write(pack(normalize_order(row, manifest)) + b"\n")
                if rows:
                    key = stage_prefix + filename
                    s3.upload_file(local, BUCKET_NAME, key, ExtraArgs={"ContentType": "application/x-ndjson"})
                    paths.append(f"s3a://{BUCKET_NAME}/{key}")
                total += len(rows)
                logger.info("Landing checked page=%s rows=%s accumulated=%s", page, len(rows), total)
    if total != manifest["count"]:
        raise ValueError("Tổng số dòng Landing không khớp manifest")
    if get_json(DAY_PREFIX + "/manifest.json") != manifest:
        raise ValueError("Manifest bị thay thế khi Bronze đang đọc; dừng trước khi ghi")
    return paths

def bronze_frame(paths):
    business = [T.StructField(name, T.StringType(), True) for name in BUSINESS_FIELDS]
    suffix = [T.StructField(name, T.StringType(), True) for name in
              ("raw_record", "data_date", "ingest_time", "run_id", "landing_run_id")]
    schema = T.StructType(business + suffix)
    if paths:
        frame = spark.read.schema(schema).option("mode", "FAILFAST").json(paths)
    else:
        frame = spark.createDataFrame([], schema)
    return frame.withColumn("data_date", F.to_date("data_date")) \
                .withColumn("ingest_time", F.to_timestamp("ingest_time"))

def write_to_iceberg(spark_session, df_data, iceberg_full_tableName, ETL_Date):
    spark_session.sql(f"CREATE NAMESPACE IF NOT EXISTS {ICEBERG_CATALOG1}.vlp")
    ddl = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in df_data.schema)
    spark_session.sql(f"CREATE TABLE IF NOT EXISTS {iceberg_full_tableName} ({ddl}) USING iceberg "
        "PARTITIONED BY (data_date) TBLPROPERTIES ('write.target-file-size-bytes'='134217728', 'write.distribution-mode'='none')")
    if spark_session.table(iceberg_full_tableName).schema.simpleString() != df_data.schema.simpleString():
        raise ValueError("Schema bảng khác cấu hình; cần review migration")
    df_data.writeTo(iceberg_full_tableName).overwrite(F.col("data_date") == F.lit(BATCH_DATE))

def write_pipeline_log(started, status, stage, count, landing_run_id, error_type=None):
    message = json.dumps({"data_date": ETL_Date, "run_id": RUN_ID, "landing_run_id": landing_run_id,
                          "stage": stage, "error_type": error_type}, ensure_ascii=False)
    schema = "target_table string, source_table string, log_message string, status string, start_date timestamp, end_date timestamp, step string, record_count long"
    log = spark.createDataFrame([(TABLE_NAME, SOURCE_NAME, message, status, started,
                                 datetime.now(timezone.utc), STEP, count)], schema)
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {ICEBERG_CATALOG1}.pipeline")
    ddl = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in log.schema)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {LOG_TABLE} ({ddl}) USING iceberg")
    log.writeTo(LOG_TABLE).append()


## Chạy Landing → Bronze

In [ ]:
globals().pop("BRONZE_RESULT", None)
RUN_ID = uuid.uuid4().hex
start_date = datetime.now(timezone.utc)
status, stage, record_count, landing_run_id, committed = "Fail", "landing_read", 0, None, False
s3, spark_ready = None, False
logger.info("START VLP Landing -> Bronze table=%s date=%s run_id=%s", TABLE_NAME, ETL_Date, RUN_ID)
try:
    s3 = init_s3()
    manifest = resolve_manifest()
    landing_run_id = manifest["run_id"]
    paths = stage_landing(manifest)
    stage = "spark_init"
    spark = init_spark()
    spark_ready = True
    spark.conf.set("spark.sql.session.timeZone", "Asia/Ho_Chi_Minh")
    df = bronze_frame(paths)
    record_count = df.count()
    if record_count != manifest["count"]: raise ValueError("Số dòng Spark không khớp manifest")
    stage = "bronze_write"
    write_to_iceberg(spark, df, TABLE_NAME, ETL_Date)
    committed, stage = True, "bronze_committed"
    logger.info("BRONZE COMMITTED rows=%s", record_count)
    write_pipeline_log(start_date, "Success", stage, record_count, landing_run_id)
    status = "Success"
    BRONZE_RESULT = {"count": record_count, "run_id": RUN_ID, "landing_run_id": landing_run_id}
    logger.info("SUCCESS Bronze và log đã ghi xong")
except Exception as exc:
    log_message = error_summary(exc)
    logger.error("FAIL stage=%s bronze_committed=%s error=%s", stage, committed, log_message)
    if spark_ready and not committed:
        try: write_pipeline_log(start_date, "Fail", stage, record_count, landing_run_id, type(exc).__name__)
        except Exception as log_exc: logger.error("Ghi log Iceberg cũng lỗi: %s", type(log_exc).__name__)
    raise RuntimeError(f"VLP thất bại stage={stage}; bronze_committed={committed}; {log_message}") from exc
finally:
    if s3 is not None:
        try:
            put_json(f"{LOG_PREFIX}/bronze-{RUN_ID}.json", {"source": "vlp", "object": "order_detail",
                "data_date": ETL_Date, "run_id": RUN_ID, "landing_run_id": landing_run_id,
                "status": status, "stage": stage, "bronze_committed": committed,
                "record_count": record_count, "error": log_message})
        except Exception as log_exc: logger.error("Ghi log S3 thất bại: %s", type(log_exc).__name__)
        try: delete_prefix(f"vlp/order_detail/_bronze_stage/{RUN_ID}/")
        except Exception as cleanup_exc: logger.warning("Cần dọn stage run=%s; error=%s", RUN_ID, type(cleanup_exc).__name__)


In [ ]:
if "BRONZE_RESULT" not in globals():
    print("Chưa có lần chạy Bronze và ghi log thành công; xem lỗi phía trên")
else:
    batch = spark.table(TABLE_NAME).filter(F.col("data_date") == F.lit(BATCH_DATE))
    batch.groupBy("data_date", "run_id", "landing_run_id").count().show(truncate=False)
    actual = batch.count()
    print("Số dòng Bronze:", actual, "| kỳ vọng:", BRONZE_RESULT["count"])
    if actual != BRONZE_RESULT["count"]: raise ValueError("Số dòng Bronze không khớp lần ghi")
    spark.table(LOG_TABLE).filter(F.col("log_message").contains(RUN_ID)).show(10, truncate=False)


## Nguồn thiết kế
- Bảng khảo sát: https://docs.google.com/spreadsheets/d/1k3VU08FAk2gbAm-cUYudu0pMfcZZWGG4/edit?gid=168101360#gid=168101360 — dòng VLP, spec liên kết tại cột R.
- API spec: https://docs.google.com/document/d/1MHpi_8CDwkOmR8MEQIkl0QkiS1RXxf8a/edit — API 1.45, vnp.eoms.order.page.detail.sub.
